# Avito Services: Candidate Generation + CatBoost Reranker

# Исправления относительно предыдущей версии

1. GEO_W 0.10 -> 0.50: топ-решение использует LLR где 0.10 * LLR_max ≈ 0.50.
   Мы использовали линейный decay [0,1] с 0.10 — в 5 раз слабее dense.
   Это приводило к тому, что семантически похожее объявление из другого города
   обгоняло ближайший релевантный. Исправлено.

2. Fine-tuning на ALL train парах: было 27K (только VALID_IDS), стало 200K.

3. CatBoost Reranker: берет топ-200 кандидатов, считает 17 признаков
   (расстояние, combined rank, dense, geo, exact match, coverages, popularity...),
   дообучается на train парах, дает +0.02-0.04 к Recall.

## Установка зависимостей

In [1]:
!pip install -q rank-bm25 sentence-transformers faiss-cpu PyStemmer pyarrow tqdm catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 99.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 747.7/747.7 kB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.8 MB/s eta 0:00:00


## Загрузка данных с Яндекс.Диска

In [2]:
import os, requests
from pathlib import Path

PUBLIC_URL = "https://disk.yandex.ru/d/sNhfo0YOjGtufg"
print("Получаем прямую ссылку...")
response = requests.get(
    f"https://cloud-api.yandex.net/v1/disk/public/resources/download?public_key={PUBLIC_URL}"
)
if response.status_code != 200:
    raise RuntimeError(f"Ошибка: {response.status_code}")
download_url = response.json()["href"]
print("Прямая ссылка получена!")

os.system(f'wget -q --show-progress -O avito_data.zip "{download_url}"')
DATA_DIR = Path("/content/first"); DATA_DIR.mkdir(exist_ok=True)
os.system(f"unzip -q -o avito_data.zip -d {DATA_DIR}")
DATA_DIR = Path("/content/data"); DATA_DIR.mkdir(exist_ok=True)
os.system(f"unzip -q -o /content/first/NLP_avito_interns/dataset.zip -d {DATA_DIR}")
for p in list(DATA_DIR.rglob("*.parquet")):
    t = DATA_DIR / p.name
    if not t.exists(): p.rename(t)
print("Данные готовы:"); os.system(f"ls -lh {DATA_DIR}/*.parquet")

Получаем прямую ссылку...
Прямая ссылка получена!
Данные готовы:


0

## Импорты и конфигурация

In [3]:
import re, pickle, time, json
from math import radians, sin, cos, sqrt, atan2
from collections import defaultdict, Counter
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

try:
    from Stemmer import Stemmer as SnowballStemmer
    _stemmer = SnowballStemmer("russian"); HAS_STEM = True
    print("[OK] PyStemmer")
except ImportError:
    HAS_STEM = False; print("[INFO] Без стемминга")

try:
    from sentence_transformers import SentenceTransformer, InputExample, losses
    from torch.utils.data import DataLoader
    import faiss
    HAS_DENSE = True; print("[OK] sentence-transformers + faiss")
except ImportError:
    HAS_DENSE = False; print("[INFO] Dense недоступен")

try:
    from catboost import CatBoostClassifier
    HAS_CB = True; print("[OK] CatBoost")
except ImportError:
    HAS_CB = False; print("[INFO] CatBoost недоступен (pip install catboost)")

SEED = 42; DATA_DIR = Path("/content/data"); SAVE_DIR = Path("/content/artifacts")
SAVE_DIR.mkdir(exist_ok=True)
N_CANDS = 50; DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

FIELD_W = {"title": 20.0, "desc": 1.0}
FIELD_B = {"title": 0.6,  "desc": 0.75}
BM25F_K1 = 1.2

# GEO_W=0.50 с linear [0,1] = эквивалент 0.10 * LLR_max в топ-решении
GEO_W = 0.50; GEO_MAX = 30.0

DENSE_MODEL = "intfloat/multilingual-e5-small"
DENSE_EPOCHS = 2; DENSE_BATCH = 64; ENC_BATCH = 256
DENSE_TOP_K = 1000  # 1000 vs 200 поднимает потолок на +0.007
DENSE_W = 0.50

LU_CAT_W = 3.0; LU_LOC_W = 2.0; LU_GL_W = 1.5

CB_POOL_SIZE    = 200   # кандидатов для реранкера
CB_MAX_TRAIN_Q  = 8000  # max train-запросов для обучения (больше = дольше)
CB_ITERATIONS   = 400; CB_DEPTH = 6
N_FEATURES = 17  # признаков для CatBoost

np.random.seed(SEED); torch.manual_seed(SEED)

[OK] PyStemmer


/tmp/ipykernel_2999/1513771205.py:18: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses


[OK] sentence-transformers + faiss
[OK] CatBoost
Device: cuda


## Загрузка данных

In [4]:
print("Загружаем данные...")
train             = pd.read_parquet(DATA_DIR / "train.parquet")
benchmark_queries = pd.read_parquet(DATA_DIR / "benchmark_queries.parquet")
benchmark_items   = pd.read_parquet(DATA_DIR / "benchmark_items.parquet")
for df, col in [(benchmark_items,"item_id"),(benchmark_queries,"query_id"),(train,"item_id")]:
    df[col] = df[col].astype(str)
VALID_IDS = set(benchmark_items["item_id"])
ALL_IDS   = benchmark_items["item_id"].tolist()
ID_TO_IDX = {iid: i for i, iid in enumerate(ALL_IDS)}
print(f"train: {len(train):,} | items: {len(benchmark_items):,} | queries: {len(benchmark_queries):,}")

Загружаем данные...
train: 497,673 | items: 189,212 | queries: 2,452


## Предобработка текста

In [5]:
def tokenize(text):
    if not text or (isinstance(text, float) and np.isnan(text)): return []
    text = re.sub(r"[^а-яa-z0-9\s]", " ", str(text).lower())
    tokens = [t for t in text.split() if len(t) >= 2]
    if HAS_STEM and tokens: tokens = _stemmer.stemWords(tokens)
    return tokens

def item_fields(row):
    """title + desc[:500]. infm_params исключен (снижает Recall по экспериментам)."""
    out = {}
    t = row.get("item_title_raw","")
    if t and not (isinstance(t,float) and np.isnan(t)): out["title"] = tokenize(str(t))
    d = row.get("item_description_raw","")
    if d and not (isinstance(d,float) and np.isnan(d)): out["desc"]  = tokenize(str(d)[:500])
    return out

def query_toks(row):
    q = row.get("search_query","")
    if q and not (isinstance(q,float) and np.isnan(q)): return tokenize(str(q))
    return []

def item_text_dense(row):
    parts = []
    t = row.get("item_title_raw",""); d = row.get("item_description_raw","")
    if t and not (isinstance(t,float) and np.isnan(t)): parts.append(str(t))
    if d and not (isinstance(d,float) and np.isnan(d)): parts.append(str(d)[:300])
    return " ".join(parts)

## BM25F

In [6]:
class BM25F:
    """
    BM25F: раздельная нормировка полей + обратный индекс.
    wtf(t,d) = sum_f w_f * tf(t,d_f)/(1 - b_f + b_f*|d_f|/avgdl_f)
    score = sum_{t in q} IDF(t) * wtf(t,d)/(k1 + wtf(t,d))
    """
    def __init__(self, field_docs, field_weights, field_b, k1=1.2, idf_floor=0.25):
        self.k1 = k1; self.n = len(field_docs); fields = list(field_weights.keys())
        avgdl = {f: sum(len(d.get(f,[])) for d in field_docs)/max(self.n,1) for f in fields}
        term_wtf = defaultdict(lambda: defaultdict(float))
        term_df  = defaultdict(set)
        for doc_id, doc in enumerate(tqdm(field_docs, desc="  BM25F", leave=False)):
            for f in fields:
                tokens = doc.get(f,[])
                if not tokens: continue
                denom = 1.0 - field_b[f] + field_b[f]*len(tokens)/avgdl[f]
                w = field_weights[f]
                for term, tf in Counter(tokens).items():
                    term_wtf[term][doc_id] += w*tf/denom
                    term_df[term].add(doc_id)
        n = self.n
        self.idf = {t: max(np.log((n-len(d)+0.5)/(len(d)+0.5)), idf_floor) for t,d in term_df.items()}
        self.inv = {}
        for term, dw in term_wtf.items():
            ids  = np.array(list(dw.keys()), dtype=np.int32)
            wtfs = np.array(list(dw.values()), dtype=np.float32)
            self.inv[term] = (ids, wtfs)
        print(f"  BM25F: {self.n:,} docs, {len(self.inv):,} terms, "
              f"avgdl title={avgdl.get('title',0):.1f} desc={avgdl.get('desc',0):.1f}")

    def get_scores_sparse(self, qtoks):
        acc = {}; k1 = self.k1
        for term in set(qtoks):
            if term not in self.inv: continue
            idf = self.idf.get(term, 0.0)
            if idf <= 0: continue
            ids, wtfs = self.inv[term]
            cs = idf * wtfs / (k1 + wtfs)
            for did, c in zip(ids.tolist(), cs.tolist()):
                acc[did] = acc.get(did, 0.0) + c
        if not acc: return np.empty(0,dtype=np.int32), np.empty(0,dtype=np.float32)
        ids = np.array(list(acc.keys()), dtype=np.int32)
        scr = np.array(list(acc.values()), dtype=np.float32)
        return ids, scr

## Гео: Haversine + центроиды локаций

In [7]:
def _hav(lat1, lon1, lat2, lon2):
    R = 6371.0; dlat = radians(lat2-lat1); dlon = radians(lon2-lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return 2*R*atan2(sqrt(a), sqrt(1-a))

print("Строим гео-кеши...")
_la = pd.to_numeric(benchmark_items["item_latitude"],  errors="coerce")
_lo = pd.to_numeric(benchmark_items["item_longitude"], errors="coerce")
_ok = ~_la.isna() & ~_lo.isna()
item_geo = dict(zip(benchmark_items.loc[_ok,"item_id"], zip(_la[_ok].values, _lo[_ok].values)))
loc_cen  = {}
if "item_location_id" in benchmark_items.columns:
    _tmp = benchmark_items.assign(_la=_la, _lo=_lo).dropna(subset=["_la","_lo"])
    for loc, g in _tmp.groupby("item_location_id"):
        loc_cen[int(loc)] = (float(g["_la"].median()), float(g["_lo"].median()))
item_location_id_map = {}
if "item_location_id" in benchmark_items.columns:
    item_location_id_map = dict(zip(
        benchmark_items["item_id"], benchmark_items["item_location_id"].fillna(-1).astype(int)
    ))
del _la, _lo, _ok, _tmp
print(f"  item_geo: {len(item_geo):,}, loc_cen: {len(loc_cen):,}")

def geo_bonus(q_loc, item_id):
    if q_loc not in loc_cen or item_id not in item_geo: return 0.0
    qlat, qlon = loc_cen[q_loc]; ilat, ilon = item_geo[item_id]
    return max(0.0, 1.0 - _hav(qlat, qlon, ilat, ilon) / GEO_MAX)

def _get_loc(q_row):
    v = q_row.get("search_location_id", None)
    if v is None or (isinstance(v,float) and np.isnan(v)): return -1
    return int(v)

Строим гео-кеши...
  item_geo: 189,211, loc_cen: 2,877


## Train Lookup + Popularity

In [8]:
print("Строим train lookup...")
valid_train = train[train["item_id"].isin(VALID_IDS)].copy()
valid_train["_q"]   = valid_train["search_query"].fillna("").str.lower().str.strip()
valid_train["_cat"] = valid_train["search_category"].fillna("").astype(str) if "search_category" in valid_train.columns else ""
valid_train["_loc"] = valid_train["search_location_id"].fillna(-1).astype(int).astype(str) if "search_location_id" in valid_train.columns else "-1"

prod_cat = defaultdict(Counter); prod_loc = defaultdict(Counter); prod_gl = defaultdict(Counter)
for _, r in tqdm(valid_train.iterrows(), total=len(valid_train), desc="  lookup", leave=False):
    q = r["_q"]; iid = r["item_id"]
    prod_cat[(q, r["_cat"])][iid] += 1
    if r["_loc"] != "-1": prod_loc[(q, r["_loc"])][iid] += 1
    prod_gl[q][iid] += 1

item_pop = Counter()
for ctr in prod_gl.values(): item_pop.update(ctr)
item_known_set = set(item_pop.keys())

q_meta = valid_train.groupby("search_query").first()[
    [c for c in ["search_location_id","search_category"] if c in valid_train.columns]
].to_dict("index")
print(f"  (q,cat): {len(prod_cat):,} | (q,loc): {len(prod_loc):,} | q: {len(prod_gl):,}")

Строим train lookup...


  lookup:   0%|          | 0/33010 [00:00<?, ?it/s]

  (q,cat): 12,209 | (q,loc): 24,307 | q: 12,208


## BM25F: токенизация и построение

In [9]:
print("Токенизируем и строим BM25F...")
t0 = time.time()
field_docs = [item_fields(r) for _, r in tqdm(benchmark_items.iterrows(),
              total=len(benchmark_items), desc="  tokenize", leave=False)]
bm25f = BM25F(field_docs, FIELD_W, FIELD_B, k1=BM25F_K1)
print(f"  Время: {time.time()-t0:.1f}с")
with open(SAVE_DIR/"bm25f.pkl","wb") as f: pickle.dump(bm25f, f)
print(f"  -> bm25f.pkl ({(SAVE_DIR/'bm25f.pkl').stat().st_size/1024/1024:.0f} MB)")

Токенизируем и строим BM25F...


  tokenize:   0%|          | 0/189212 [00:00<?, ?it/s]

  BM25F:   0%|          | 0/189212 [00:00<?, ?it/s]

  BM25F: 189,212 docs, 196,287 terms, avgdl title=4.3 desc=50.7
  Время: 54.5с
  -> bm25f.pkl (89 MB)


## Dense Retrieval: fine-tuning + encode + FAISS

In [10]:
dense_model = None; item_emb_matrix = None; faiss_index = None

if HAS_DENSE:
    print(f"\nЗагружаем {DENSE_MODEL}...")
    dense_model = SentenceTransformer(DENSE_MODEL, device=DEVICE)
    emb_dim     = dense_model.get_sentence_embedding_dimension()
    print(f"  dim={emb_dim}, params={sum(p.numel() for p in dense_model.parameters()):,}")

    if torch.cuda.is_available():
        print("\nFine-tuning (in-batch negatives, ALL train)...")
        print("  Hard negatives НЕ используем: одинаковый заголовок в другой локации")
        print("  — текст не различает, различает гео. Противоречивый сигнал для энкодера.")
        pairs_src = train.drop_duplicates(subset=["search_query","item_id"])
        q_texts   = pairs_src["search_query"].fillna("").astype(str).tolist()
        def _txt(row):
            parts = []
            t = row.get("item_title_raw",""); d = row.get("item_description_raw","")
            if t and not(isinstance(t,float) and np.isnan(t)): parts.append(str(t))
            if d and not(isinstance(d,float) and np.isnan(d)): parts.append(str(d)[:300])
            return " ".join(parts)
        i_texts = [_txt(r) for _,r in tqdm(pairs_src.iterrows(), total=len(pairs_src), desc="  item texts", leave=False)]
        MAX_PAIRS = 200_000
        examples = [InputExample(texts=["query: "+q, "passage: "+i])
                    for q,i in zip(q_texts[:MAX_PAIRS], i_texts[:MAX_PAIRS]) if q.strip() and i.strip()]
        print(f"  Training pairs: {len(examples):,}")
        loader  = DataLoader(examples, shuffle=True, batch_size=DENSE_BATCH, drop_last=True)
        loss_fn = losses.MultipleNegativesRankingLoss(dense_model)
        dense_model.fit(
            train_objectives=[(loader, loss_fn)],
            epochs=DENSE_EPOCHS, warmup_steps=int(len(loader)*DENSE_EPOCHS*0.05),
            show_progress_bar=True, output_path=str(SAVE_DIR/"dense_model_ft"),
            use_amp=True,
            checkpoint_save_steps=1000,    # ← сохранять каждые 1000 шагов
    checkpoint_path=str(SAVE_DIR/"checkpoints"),
        )
        dense_model = SentenceTransformer(str(SAVE_DIR/"dense_model_ft"), device=DEVICE)
        print("  Fine-tuning завершен")
    else:
        print("  GPU не найден — pre-trained (для fine-tuning нужна T4 GPU)")

    print(f"\nКодируем {len(benchmark_items):,} объявлений...")
    t0 = time.time()
    item_emb_matrix = dense_model.encode(
        ["passage: " + item_text_dense(r) for _,r in benchmark_items.iterrows()],
        batch_size=ENC_BATCH, normalize_embeddings=True,
        show_progress_bar=True, convert_to_numpy=True, device=DEVICE,
    ).astype("float32")
    faiss_index = faiss.IndexFlatIP(emb_dim); faiss_index.add(item_emb_matrix)
    np.save(SAVE_DIR/"item_emb.npy", item_emb_matrix)
    faiss.write_index(faiss_index, str(SAVE_DIR/"faiss.bin"))
    print(f"  {item_emb_matrix.shape}, {time.time()-t0:.1f}с -> item_emb.npy + faiss.bin")
else:
    print("[WARN] Dense недоступен")


Загружаем intfloat/multilingual-e5-small...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  Fine-tuning завершен

Кодируем 189,212 объявлений...


Batches:   0%|          | 0/740 [00:00<?, ?it/s]

  (189212, 384), 662.8с -> item_emb.npy + faiss.bin


## Helper: пул кандидатов

In [11]:
def get_pool(q_norm, q_cat, q_loc, dense_top_idx=None, dense_top_scores=None,
             lu_gl=None, lu_cat=None, lu_loc=None, pool_size=CB_POOL_SIZE):
    """BM25F (все ненулевые) + Dense (top-K) + Lookup -> top-pool_size."""
    if lu_gl is None: lu_gl, lu_cat, lu_loc = prod_gl, prod_cat, prod_loc
    qtoks = tokenize(q_norm)
    bm25_d = {}; dense_d = {}; combined = {}

    if qtoks:
        doc_ids, raw_s = bm25f.get_scores_sparse(qtoks)
        if len(doc_ids) > 0:
            max_s = float(raw_s.max()); max_s = max_s if max_s > 0 else 1.0
            for did, ns in zip(doc_ids.tolist(), (raw_s/max_s).tolist()):
                iid = ALL_IDS[did]; g = geo_bonus(q_loc, iid) if q_loc != -1 else 0.0
                bm25_d[iid] = ns; combined[iid] = ns + GEO_W * g

    if dense_top_idx is not None:
        max_ds = float(dense_top_scores[0]) if dense_top_scores[0] > 0 else 1.0
        for ds, di in zip(dense_top_scores, dense_top_idx):
            if di < 0: continue
            iid = ALL_IDS[di]; dn = float(ds)/max_ds; dense_d[iid] = dn
            combined[iid] = combined.get(iid, 0.0) + DENSE_W * dn

    for ctr, w in [(lu_cat.get((q_norm, q_cat),{}), LU_CAT_W),
                   (lu_loc.get((q_norm, str(q_loc)),{}), LU_LOC_W),
                   (lu_gl.get(q_norm, {}), LU_GL_W)]:
        if not ctr: continue
        mx = max(ctr.values())
        for iid, cnt in ctr.items():
            if iid in VALID_IDS: combined[iid] = combined.get(iid, 0.0) + w*cnt/mx

    pool_ids = [iid for iid,_ in sorted(combined.items(), key=lambda x:-x[1]) if iid in VALID_IDS][:pool_size]
    return pool_ids, bm25_d, dense_d, qtoks

## CatBoost: вычисление признаков

In [12]:
FEATURE_NAMES = ["dist_km","combined_score","combined_rank","dense_score","geo_score",
                 "loc_exact","bm25f_score","geo_rank","bm25f_rank","desc_cov",
                 "idf_title_cov","title_cov","popularity","dist_rank","known","dense_rank","pool_size"]
assert len(FEATURE_NAMES) == N_FEATURES

def compute_features(pool_ids, bm25_d, dense_d, qtoks, q_loc):
    n = len(pool_ids)
    if n == 0: return np.empty((0, N_FEATURES), dtype=np.float32)
    q_set = set(qtoks); total_idf = sum(bm25f.idf.get(t, 0.0) for t in qtoks)

    bm25_a  = np.array([bm25_d.get(iid, 0.0) for iid in pool_ids], dtype=np.float32)
    dense_a = np.array([dense_d.get(iid, 0.0) for iid in pool_ids], dtype=np.float32)

    if q_loc in loc_cen:
        qlat, qlon = loc_cen[q_loc]
        dist_a = np.array([_hav(qlat, qlon, *item_geo[iid]) if iid in item_geo else GEO_MAX
                           for iid in pool_ids], dtype=np.float32)
    else:
        dist_a = np.full(n, GEO_MAX, dtype=np.float32)
    geo_a = np.maximum(0.0, 1.0 - dist_a / GEO_MAX)

    loc_match = np.array([float(item_location_id_map.get(iid,-2)==q_loc and q_loc!=-1)
                          for iid in pool_ids], dtype=np.float32)
    combined_a = bm25_a + GEO_W * geo_a + DENSE_W * dense_a

    tc_list = []; dc_list = []; ic_list = []
    for iid in pool_ids:
        did = ID_TO_IDX.get(iid, -1)
        if did < 0 or not q_set:
            tc_list.append(0.0); dc_list.append(0.0); ic_list.append(0.0); continue
        tt = set(field_docs[did].get("title", [])); dt = set(field_docs[did].get("desc", []))
        tc_list.append(len(q_set & tt) / len(q_set))
        dc_list.append(len(q_set & dt) / len(q_set))
        ic_list.append(sum(bm25f.idf.get(t,0.0) for t in qtoks if t in tt) / max(total_idf, 1e-10))
    tc = np.array(tc_list, dtype=np.float32)
    dc = np.array(dc_list, dtype=np.float32)
    ic = np.array(ic_list, dtype=np.float32)

    pop_a   = np.array([float(item_pop.get(iid,0)) for iid in pool_ids], dtype=np.float32)
    known_a = np.array([float(iid in item_known_set) for iid in pool_ids], dtype=np.float32)

    def rnk(arr, asc=False):
        order = np.argsort(arr) if asc else np.argsort(-arr)
        r = np.empty(n, dtype=np.float32); r[order] = np.arange(1,n+1,dtype=np.float32)/n
        return r

    return np.column_stack([
        dist_a, combined_a, rnk(combined_a), dense_a, geo_a, loc_match,
        bm25_a, rnk(geo_a), rnk(bm25_a), dc, ic, tc, pop_a,
        rnk(dist_a, asc=True), known_a, rnk(dense_a), np.full(n, float(n), dtype=np.float32),
    ]).astype(np.float32)

## OOD Validation Split

In [13]:
unique_q = valid_train["_q"].unique().copy()
np.random.seed(SEED); np.random.shuffle(unique_q)
val_q_set = set(unique_q[:500])

val_ood = (
    valid_train[valid_train["_q"].isin(val_q_set)]
    .groupby("search_query")["item_id"].apply(set).reset_index()
)
val_ood.columns = ["search_query", "relevant_ids"]

val_gl = defaultdict(Counter); val_cat = defaultdict(Counter); val_loc = defaultdict(Counter)
for _, r in valid_train[~valid_train["_q"].isin(val_q_set)].iterrows():
    q = r["_q"]; iid = r["item_id"]
    val_cat[(q,r["_cat"])][iid] += 1
    if r["_loc"] != "-1": val_loc[(q,r["_loc"])][iid] += 1
    val_gl[q][iid] += 1

print(f"OOD validation: {len(val_ood)} запросов")

OOD validation: 500 запросов


## OOD Validation: без реранкера

In [18]:
print("\nOffline val (BM25F + geo + dense, без реранкера)...")
val_d_scores = val_d_idx = None
if HAS_DENSE and dense_model is not None:
    val_q_embs = dense_model.encode(
        ["query: " + r["search_query"] for _,r in val_ood.iterrows()],
        batch_size=ENC_BATCH, normalize_embeddings=True, show_progress_bar=False,
        convert_to_numpy=True,
    ).astype("float32")
    val_d_scores, val_d_idx = faiss_index.search(val_q_embs, DENSE_TOP_K)

recalls_pre = []
for vi, (_, row) in enumerate(tqdm(val_ood.iterrows(), total=len(val_ood), desc="  val-pre", leave=False)):
    rel = {str(i) for i in row["relevant_ids"]} & VALID_IDS
    if not rel: continue
    q = row["search_query"]; meta = q_meta.get(q, {})
    ql = int(meta.get("search_location_id", -1) or -1)
    qc = str(meta.get("search_category", "") or "")
    di = val_d_idx[vi]    if val_d_idx    is not None else None
    ds = val_d_scores[vi] if val_d_scores is not None else None
    pool_ids, *_ = get_pool(q.lower().strip(), qc, ql, di, ds,
                             lu_gl=val_gl, lu_cat=val_cat, lu_loc=val_loc)
    recalls_pre.append(len(rel & set(pool_ids[:N_CANDS])) / len(rel))

r_pre = float(np.mean(recalls_pre)) if recalls_pre else 0.0
print(f"\n  Recall@{N_CANDS} [OOD, pre-rerank]: {r_pre:.4f}")
print(f"  (было 0.4040 при GEO_W=0.10; ожидаем ~0.55-0.65 при GEO_W=0.50)")


Offline val (BM25F + geo + dense, без реранкера)...


  val-pre:   0%|          | 0/500 [00:00<?, ?it/s]


  Recall@50 [OOD, pre-rerank]: 0.6344
  (было 0.4040 при GEO_W=0.10; ожидаем ~0.55-0.65 при GEO_W=0.50)


## CatBoost: построение обучающего пула

In [20]:
X_cb = np.empty((0, N_FEATURES), dtype=np.float32)
y_cb = np.empty(0, dtype=np.int32)

if HAS_CB and HAS_DENSE and dense_model is not None and faiss_index is not None:
    print("\nCatBoost: строим обучающий пул...")
    all_train_q = valid_train["search_query"].unique().tolist()
    np.random.seed(SEED); np.random.shuffle(all_train_q)
    train_q_list = all_train_q[:CB_MAX_TRAIN_Q]
    print(f"  Обучающих запросов: {len(train_q_list):,}")

    q_to_pos = {}
    for _, r in valid_train.iterrows():
        q_to_pos.setdefault(str(r["search_query"]), set()).add(r["item_id"])

    print("  Batch encode train queries...")
    tr_q_embs = dense_model.encode(
        ["query: " + q for q in train_q_list], batch_size=ENC_BATCH,
        normalize_embeddings=True, show_progress_bar=True, convert_to_numpy=True,
    ).astype("float32")
    tr_d_scores, tr_d_idx = faiss_index.search(tr_q_embs, DENSE_TOP_K)
    print("  FAISS done. Генерируем признаки...")

    X_parts = []; y_parts = []
    for qi, q_text in enumerate(tqdm(train_q_list, desc="  reranker pool")):
        meta  = q_meta.get(q_text, {})
        q_loc = int(meta.get("search_location_id", -1) or -1)
        q_cat = str(meta.get("search_category", "") or "")
        q_norm = q_text.lower().strip()
        pool_ids, bm25_d, dense_d, qtoks = get_pool(
            q_norm, q_cat, q_loc, tr_d_idx[qi], tr_d_scores[qi],
            pool_size=CB_POOL_SIZE,
        )
        if not pool_ids: continue
        feats  = compute_features(pool_ids, bm25_d, dense_d, qtoks, q_loc)
        pos    = q_to_pos.get(q_text, set())
        labels = np.array([1 if iid in pos else 0 for iid in pool_ids], dtype=np.int32)
        if labels.sum() == 0: continue
        X_parts.append(feats); y_parts.append(labels)

    if X_parts:
        X_cb = np.vstack(X_parts); y_cb = np.concatenate(y_parts)
        pos_rate = float(y_cb.mean())
        print(f"\n  Samples: {len(X_cb):,}, positives: {y_cb.sum():,} ({pos_rate*100:.2f}%)")
    else:
        print("  [WARN] Пул пуст")
else:
    print("[INFO] CatBoost или Dense недоступен — пропускаем реранкер")


CatBoost: строим обучающий пул...
  Обучающих запросов: 8,000
  Batch encode train queries...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

  FAISS done. Генерируем признаки...


  reranker pool:   0%|          | 0/8000 [00:00<?, ?it/s]


  Samples: 1,600,000, positives: 17,693 (1.11%)


## CatBoost: обучение

In [23]:
cb_model = None

if HAS_CB and len(y_cb) > 0 and y_cb.sum() > 0:
    pos_rate = float(y_cb.mean())
    pos_weight = int(round(1.0 / max(pos_rate, 0.001)))
    print(f"\nCatBoost: {len(X_cb):,} samples, pos_weight={pos_weight}, {CB_ITERATIONS} trees depth={CB_DEPTH}")

    cb_model = CatBoostClassifier(
        iterations=CB_ITERATIONS, depth=CB_DEPTH, loss_function="Logloss",
        scale_pos_weight=pos_weight, random_seed=SEED, verbose=50,
    )
    cb_model.fit(X_cb, y_cb)
    cb_model.save_model(str(SAVE_DIR/"cb_reranker.cbm"))
    print("  -> cb_reranker.cbm")

    fi = dict(zip(FEATURE_NAMES, cb_model.get_feature_importance()))
    print("\n  Топ признаков:")
    for fn, fv in sorted(fi.items(), key=lambda x:-x[1])[:12]:
        print(f"    {fn:<25} {fv:.2f}")
elif HAS_CB:
    print("[INFO] Нет обучающих данных для CatBoost")


CatBoost: 1,600,000 samples, pos_weight=90, 400 trees depth=6
Learning rate set to 0.5
0:	learn: 0.1899166	total: 555ms	remaining: 3m 41s
50:	learn: 0.0594721	total: 27.2s	remaining: 3m 6s
100:	learn: 0.0504051	total: 54.9s	remaining: 2m 42s
150:	learn: 0.0448440	total: 1m 22s	remaining: 2m 15s
200:	learn: 0.0406346	total: 1m 50s	remaining: 1m 49s
250:	learn: 0.0374131	total: 2m 18s	remaining: 1m 22s
300:	learn: 0.0346416	total: 2m 45s	remaining: 54.3s
350:	learn: 0.0321198	total: 3m 11s	remaining: 26.8s
399:	learn: 0.0300169	total: 3m 39s	remaining: 0us
  -> cb_reranker.cbm

  Топ признаков:
    popularity                35.69
    geo_rank                  13.44
    known                     13.11
    combined_rank             12.27
    dist_km                   4.74
    dist_rank                 3.82
    geo_score                 3.20
    bm25f_rank                2.48
    dense_rank                2.01
    idf_title_cov             1.86
    combined_score            1.78
    dense_

## OOD Validation: с реранкером

In [24]:
r_post = r_pre  # default если нет реранкера

if cb_model is not None:
    print("\nOffline val (с реранкером)...")
    recalls_post = []
    for vi, (_, row) in enumerate(tqdm(val_ood.iterrows(), total=len(val_ood),
                                        desc="  val-post", leave=False)):
        rel = {str(i) for i in row["relevant_ids"]} & VALID_IDS
        if not rel: continue
        q = row["search_query"]; meta = q_meta.get(q, {})
        ql = int(meta.get("search_location_id", -1) or -1)
        qc = str(meta.get("search_category", "") or "")
        di = val_d_idx[vi]    if val_d_idx    is not None else None
        ds = val_d_scores[vi] if val_d_scores is not None else None
        pool_ids, bm25_d, dense_d, qtoks = get_pool(
            q.lower().strip(), qc, ql, di, ds,
            lu_gl=val_gl, lu_cat=val_cat, lu_loc=val_loc,
            pool_size=CB_POOL_SIZE,
        )
        if not pool_ids:
            recalls_post.append(0.0); continue
        feats = compute_features(pool_ids, bm25_d, dense_d, qtoks, ql)
        probs = cb_model.predict_proba(feats)[:, 1]
        top50 = [pool_ids[j] for j in np.argsort(-probs)][:N_CANDS]
        recalls_post.append(len(rel & set(top50)) / len(rel))

    r_post = float(np.mean(recalls_post)) if recalls_post else 0.0
    print(f"\n  Recall@{N_CANDS} [OOD, pre-rerank]:  {r_pre:.4f}")
    print(f"  Recall@{N_CANDS} [OOD, post-rerank]: {r_post:.4f}  ({(r_post-r_pre)*100:+.2f} п.п.)")


Offline val (с реранкером)...


  val-post:   0%|          | 0/500 [00:00<?, ?it/s]


  Recall@50 [OOD, pre-rerank]:  0.6344
  Recall@50 [OOD, post-rerank]: 0.7650  (+13.06 п.п.)


## Генерация answer.csv

In [25]:
print(f"\nГенерируем {len(benchmark_queries):,} запросов...")

bench_d_scores = bench_d_idx = None
if HAS_DENSE and dense_model is not None:
    print("  Batch encode benchmark queries...")
    bench_q_embs = dense_model.encode(
        ["query: " + str(r.get("search_query","")) for _,r in benchmark_queries.iterrows()],
        batch_size=ENC_BATCH, normalize_embeddings=True,
        show_progress_bar=True, convert_to_numpy=True,
    ).astype("float32")
    bench_d_scores, bench_d_idx = faiss_index.search(bench_q_embs, DENSE_TOP_K)

all_qids = []; all_preds = []
for i, (_, q_row) in enumerate(tqdm(benchmark_queries.iterrows(),
                                     total=len(benchmark_queries), desc="queries")):
    qid   = str(q_row["query_id"])
    q     = str(q_row.get("search_query","")).lower().strip()
    q_cat = str(q_row.get("search_category",""))
    q_loc = _get_loc(q_row)
    di    = bench_d_idx[i]    if bench_d_idx    is not None else None
    ds    = bench_d_scores[i] if bench_d_scores is not None else None

    pool_ids, bm25_d, dense_d, qtoks = get_pool(q, q_cat, q_loc, di, ds, pool_size=CB_POOL_SIZE)

    if cb_model is not None and pool_ids:
        feats = compute_features(pool_ids, bm25_d, dense_d, qtoks, q_loc)
        probs = cb_model.predict_proba(feats)[:, 1]
        top50 = [pool_ids[j] for j in np.argsort(-probs)][:N_CANDS]
    else:
        top50 = pool_ids[:N_CANDS]

    all_qids.append(qid)
    all_preds.append([iid for iid in top50 if iid in VALID_IDS][:N_CANDS])


Генерируем 2,452 запросов...
  Batch encode benchmark queries...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

queries:   0%|          | 0/2452 [00:00<?, ?it/s]

## Сохранение и проверка

In [26]:
answer = pd.DataFrame({"query_id": all_qids, "answer": [" ".join(p) for p in all_preds]})
answer.to_csv("answer.csv", index=False, encoding="utf-8")
print(f"Сохранено: answer.csv ({len(answer)} строк)")

check = pd.read_csv("answer.csv", dtype=str)
assert list(check.columns) == ["query_id","answer"]
assert set(check["query_id"]) == set(benchmark_queries["query_id"].astype(str))
assert check["query_id"].nunique() == len(check)
check["items"] = check["answer"].str.split(" ")
check["n"]     = check["items"].apply(lambda x: len([i for i in x if i]) if isinstance(x,list) else 0)
assert (check["n"] <= N_CANDS).all()
dupes = check["items"].apply(lambda x: len([i for i in x if i]) != len({i for i in x if i}) if isinstance(x,list) else False)
assert not dupes.any()
print("[OK] Все проверки прошли")

print(f"\n=== ИТОГ ===")
print(f"  OOD Recall (без реранкера):  {r_pre:.4f}")
print(f"  OOD Recall (с реранкером):   {r_post:.4f}")
print(f"  Строк в answer.csv:          {len(check)}")
print(f"  Ср. кандидатов/запрос:       {check['n'].mean():.2f}")
print(f"\nanswer.csv готов!")

Сохранено: answer.csv (2452 строк)
[OK] Все проверки прошли

=== ИТОГ ===
  OOD Recall (без реранкера):  0.6344
  OOD Recall (с реранкером):   0.7650
  Строк в answer.csv:          2452
  Ср. кандидатов/запрос:       50.00

answer.csv готов!
